# dispatch-back-fn-from-recipe — ex2: dispatch with friendly KeyError naming op + argnum + fix

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dispatch-back-fn-from-recipe`. Running the final beacon cell reports progress against the `Backprop: dispatch back fn from recipe` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: dispatch back fn from recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dispatch-back-fn-from-recipe`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dispatch-back-fn-from-recipe"
DD_SUBTOPIC = "Backprop: dispatch back fn from recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Dispatch back_fn with friendly KeyError — quick refresher

When `(recipe.func, argnum)` isn't in the registry, the raw `back_funcs[key]` lookup raises a `KeyError` that prints the tuple but tells you nothing about WHICH op or argnum is unregistered. Wrap the dispatch with a diagnostic:

```python
def dispatch(node, back_funcs):
    out = []
    for argnum, parent in node.recipe.parents.items():
        key = (node.recipe.func, argnum)
        if key not in back_funcs:
            fn_name = getattr(node.recipe.func, '__name__', repr(node.recipe.func))
            raise KeyError(
                f'No back_fn registered for ({fn_name}, argnum={argnum}). '
                f'Add it to BACK_FUNCS via register_back_func({fn_name}, {argnum}, ...).'
            )
        out.append((argnum, parent, back_funcs[key]))
    return out
```

The naming pattern matches PyTorch's autograd: error messages that tell you exactly what was missing AND how to fix it — not just `KeyError: (<built-in function sin>, 0)`.

### Exercise 2 — dispatch with friendly KeyError naming op + argnum + fix

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the (recipe.func, argnum)-keyed dispatch pattern while wrapping the registry lookup with a custom KeyError that names the forward fn AND the argnum that has no registered back_fn — so users see actionable error messages instead of opaque tuples.
> Keywords: dispatch, diagnostic, key-error, error-message
> ```

**KCs targeted:** `dispatch-back-fn-from-recipe`, `parents-dict-by-argidx`

Implement `dispatch_back_fns_diag(node, back_funcs)` — same return shape as ex1 (`[(argnum, parent, back_fn), ...]`), but when a `(recipe.func, argnum)` key is MISSING from `back_funcs`, raise a `KeyError` whose message names:

1. The forward function (use `__name__` if available, else `repr(fn)`).
2. The argnum that's missing.
3. A suggested fix: `'register_back_func({fn_name}, {argnum}, ...)'`.

**Example message** for an unregistered `t.sin` at arg 0:
```
KeyError: No back_fn registered for (sin, argnum=0). Add it via register_back_func(sin, 0, ...).
```

**Why this matters.** The raw `KeyError: (<built-in function sin>, 0)` is technically correct but cryptic — users have to know what the tuple means. A human-readable message is the whole point of a diagnostic wrapper.

**Algorithm.**
```python
for argnum, parent in node.recipe.parents.items():
    key = (node.recipe.func, argnum)
    if key not in back_funcs:
        fn_name = getattr(node.recipe.func, '__name__', repr(node.recipe.func))
        raise KeyError(
            f'No back_fn registered for ({fn_name}, argnum={argnum}). '
            f'Add it via register_back_func({fn_name}, {argnum}, ...).'
        )
    out.append((argnum, parent, back_funcs[key]))
```

**Return type.** `list[tuple]`. **Error type.** `KeyError` (not RuntimeError, not a custom class — the caller still catches `KeyError` if it wants to handle this generically).

In [ ]:
def dispatch_back_fns_diag(node, back_funcs) -> list:
    out = []
    for argnum, parent in node.recipe.parents.items():
        key = (node.recipe.func, argnum)
        if key not in back_funcs:
            fn = node.recipe.func
            fn_name = getattr(fn, '__name__', repr(fn))
            raise KeyError(
                f'No back_fn registered for ({fn_name}, argnum={argnum}). '
                f'Add it via register_back_func({fn_name}, {argnum}, ...).'
            )
        out.append((argnum, parent, back_funcs[key]))
    return out


<details><summary>Solution</summary>

```python
def dispatch_back_fns_diag(node, back_funcs) -> list:
    out = []
    for argnum, parent in node.recipe.parents.items():
        key = (node.recipe.func, argnum)
        if key not in back_funcs:
            fn = node.recipe.func
            fn_name = getattr(fn, '__name__', repr(fn))
            raise KeyError(
                f'No back_fn registered for ({fn_name}, argnum={argnum}). '
                f'Add it via register_back_func({fn_name}, {argnum}, ...).'
            )
        out.append((argnum, parent, back_funcs[key]))
    return out
```

**Diagnostic messages are a feature, not polish.** PyTorch's autograd raises `RuntimeError: Trying to backward through the graph a second time...` — a sentence, not a tuple. Every minute the user spends decoding `KeyError: (<built-in function sin>, 0)` is a minute they're not fixing the bug. The wrapper is the API surface where you spend a paragraph of error message in exchange for hours of user-debug time.

**Why `getattr(fn, '__name__', repr(fn))`.** Built-in torch ops have `__name__`. Some wrapped/partial functions don't — the `repr` fallback keeps the message printable on any callable.

**Why still `KeyError`, not a new exception class.** Subclassing KeyError would let callers catch `KeyError` to handle 'op not registered yet' uniformly with native dict misses. Don't fragment the exception hierarchy without a reason.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()